# SimpleElasticSolid Demo

In [ ]:
import MeshFEM, mesh, mesh_energy, py_newton_optimizer, benchmark
import simple_elastic_solid, elastic_solid

### Construct objective term(s) and problem

In [ ]:
m = mesh.Mesh('../3rdparty/MeshFEM/misc/examples/meshes/bunny_coarse.msh')
x = mesh_energy.NodalVars(m, 3)

In [ ]:
es = simple_elastic_solid.SimpleElasticSolid(m.vertices(), m.elements(), x)
# es = simple_elastic_solid.SolidMeshEnergy(m, x) # MeshEnergy version
# es = simple_elastic_solid.AutodiffSolid(m, x) # Autodiff version
p = py_newton_optimizer.NewtonMultiobjectiveProblem(x, [es])

### Derivative checks

In [ ]:
import fd_validation
fd_validation.gradHessConvergencePlot(p)

### Visualization

In [ ]:
import viewer
em = MeshFEM.EmbeddedMesh(m, x)
v = viewer.Viewer(em, wireframe=True)
p.setCustomIterationCallback(v.updater()) # Update the viewer after each iteration.
v.show()

### Configure BCs: glue to ground, pull ear up.

In [ ]:
import sim_utils
groundVars = sim_utils.getBBoxVars(m, sim_utils.BBoxFace.MIN_Y, tol=4)
pullVars = sim_utils.getBBoxVars(m, sim_utils.BBoxFace.MAX_Y, tol=0.001)

xval = x.getVars()
xval[pullVars[1]] += 20
x.setVars(xval)
v.update()

p.setFixedVars(groundVars + pullVars)

### Run optimization

In [ ]:
benchmark.reset()
p.optimizer().optimize();
benchmark.report()

In [ ]:
for name, pattern in [('Non-projected Hessian Eval', 'Solid.*hessian$'), ('Projected Hessian Eval', 'Solid.*hessian \(projected\)$')]:
    print(f'{name}: {benchmark.totalTime(pattern)}s / {benchmark.numInvocations(pattern)} evals')